# PKTD3-TD: Training on Google Colab (T4 GPU)

This notebook executes the full training procedure (Algorithm 1) of **PKTD3-TD** using GPU acceleration on Google Colab.

Reference Paper:  
> *M. Li et al., "3-D Trajectory Design Based on Deep Reinforcement Learning for UAV-Assisted Communication Networks," IEEE Transactions on Network Science and Engineering, vol. 13, no. 1, pp. 248–261, 2026.*

---
### Prerequisites & Setup Instructions
1. **Enable Hardware Acceleration:**  
   In Google Colab, go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. **Store GitHub Personal Access Token (PAT):**  
   In the left sidebar, click on the **Secrets (key icon)**:  
   - Click **Add new secret**  
   - Name: `GITHUB_PAT_TOKEN`  
   - Value: `<your-github-personal-access-token>` (with repo write permissions)  
   - Toggle **Notebook access** to **ON**.
3. **Google Drive Backup:**  
   Google Drive is mounted to save checkpoints continuously in real time. If Colab disconnects or times out, your checkpoints remain safe in Drive.

In [ ]:
# Step 1: Mount Google Drive for persistent checkpoint backup
import os
from google.colab import drive

drive.mount('/content/drive')
drive_backup_dir = '/content/drive/MyDrive/PKTD3_TD_Checkpoints/run1'
os.makedirs(drive_backup_dir, exist_ok=True)
print(f"Persistent backup directory ready: {drive_backup_dir}")

In [ ]:
# Step 2: Clone repository using GITHUB_PAT_TOKEN secret
import os
from google.colab import userdata

github_token = userdata.get('GITHUB_PAT_TOKEN')
repo_owner = "Krishna200608"
repo_name = "uav_trajectory_rl"
repo_url = f"https://{github_token}@github.com/{repo_owner}/{repo_name}.git"

work_dir = f"/content/{repo_name}"
if not os.path.exists(work_dir):
    !git clone {repo_url} {work_dir}
    %cd {work_dir}
else:
    %cd {work_dir}
    !git pull

!git status

In [ ]:
# Step 3: Install dependencies and verify GPU acceleration
!pip install -e ".[dev]"

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: GPU is not enabled! Switch runtime to T4 GPU in Runtime > Change runtime type.")

# Run unit tests to confirm environment integrity
!pytest tests/ -q

In [ ]:
# Step 4: Execute PKTD3-TD Training (Algorithm 1, 6000 episodes)
# Checkpoints are saved directly to Google Drive so no progress is lost upon disconnection
import shutil

local_checkpoint_dir = "/content/uav_trajectory_rl/checkpoints/run1"
os.makedirs(local_checkpoint_dir, exist_ok=True)

!python scripts/train.py \
    --episodes 6000 \
    --k 10 \
    --batch-size 128 \
    --seed 0 \
    --checkpoint-dir "{drive_backup_dir}" \
    --checkpoint-every 500 \
    --log-every 20

# Sync all checkpoints and reward history from Drive to local repository folder
print("\nCopying saved checkpoints into local repository folder...")
shutil.copytree(drive_backup_dir, local_checkpoint_dir, dirs_exist_ok=True)
!ls -lh {local_checkpoint_dir}

In [ ]:
# Step 5: Plot and visualize reward convergence
import os
import numpy as np
import matplotlib.pyplot as plt

rewards_file = os.path.join(local_checkpoint_dir, "episode_rewards.npy")
if os.path.exists(rewards_file):
    rewards = np.load(rewards_file)
    window = 100
    moving_avg = np.convolve(rewards, np.ones(window) / window, mode="valid")

    plt.figure(figsize=(10, 5))
    plt.plot(rewards, alpha=0.3, color="steelblue", label="Episode Reward (raw)")
    plt.plot(range(window - 1, len(rewards)), moving_avg, color="crimson", linewidth=2, label=f"Moving Average ({window} eps)")
    plt.xlabel("Episode")
    plt.ylabel("Cumulative Reward")
    plt.title("PKTD3-TD Training Reward Convergence (Google Colab T4 GPU)")
    plt.grid(True, linestyle="--", alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plot_path = os.path.join(local_checkpoint_dir, "training_reward_curve.png")
    plt.savefig(plot_path, dpi=300)
    plt.show()
    print(f"Reward curve saved to {plot_path}")
else:
    print(f"Rewards file not found at {rewards_file}")

In [ ]:
# Step 6: Commit and push all checkpoints and results to GitHub
!git config --global user.name "Krishna200608"
!git config --global user.email "krishnasikheriya001@gmail.com"

# Force-add checkpoints/run1 (overriding .gitignore for saved model artifacts)
!git add -f checkpoints/run1
!git status

!git commit -m "Add PKTD3-TD trained model checkpoints and reward history from Colab T4 run"
!git push origin main
print("\nSuccessfully pushed checkpoints and reward logs to GitHub!")